In [1]:
# セル1：ライブラリのインポート
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

print("ライブラリの読み込みが完了しました。")

ライブラリの読み込みが完了しました。


In [2]:
# セル2：メインフォルダの選択

# ダイアログ用ウィンドウを裏で作成（最前面に固定）
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# フォルダ選択ダイアログを開く
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

✅ 選択されたメインフォルダ:
E:/001_2026AI班/futabaデータ/半透明PP(J707EG)/合わせたやつ(0.5mm‗再々,1.0mm_＋再)/データフォルダ


In [3]:
# セル3：対象ファイルのリストアップ（事前確認）

if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してフォルダを選んでください。")

target_files = []

# os.walkでフォルダの最深部までCSVファイルを探す
for dirpath, dirnames, filenames in os.walk(main_dir):
    for f in filenames:
        if f.lower().endswith('.csv'):
            # ファイルのフルパスを作成してリストに追加
            target_files.append(os.path.join(dirpath, f))

print(f"🔍 見つかったCSVファイル: 合計 {len(target_files)} 件")

# 最初の数件だけ試しにパスを表示して確認
if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print("   ... (他省略)")
else:
    print("⚠️ CSVファイルが見つかりませんでした。フォルダを確認してください。")

🔍 見つかったCSVファイル: 合計 1290 件

【処理対象ファイルの例】
 - E:/001_2026AI班/futabaデータ/半透明PP(J707EG)/合わせたやつ(0.5mm‗再々,1.0mm_＋再)/データフォルダ\0.5mm\190℃\001_190℃_020_0.5mm.csv
 - E:/001_2026AI班/futabaデータ/半透明PP(J707EG)/合わせたやつ(0.5mm‗再々,1.0mm_＋再)/データフォルダ\0.5mm\190℃\002_190℃_020_0.5mm.csv
 - E:/001_2026AI班/futabaデータ/半透明PP(J707EG)/合わせたやつ(0.5mm‗再々,1.0mm_＋再)/データフォルダ\0.5mm\190℃\003_190℃_020_0.5mm.csv
   ... (他省略)


In [4]:
# セル4：フーリエ変換の一括実行と画像指定フォルダ保存

if not target_files:
    raise ValueError("処理対象のCSVファイルがありません。")

# --- 画像をまとめる出力フォルダの設定 ---
# os.getcwd() でこのプログラム（Jupyter Notebook）が保存されているフォルダのパスを取得します
current_dir = os.getcwd()
output_dir_name = "fft_results"  # まとめるフォルダの名前
output_dir_path = os.path.join(current_dir, output_dir_name)

# フォルダが存在しない場合は新しく作成する
os.makedirs(output_dir_path, exist_ok=True)
# ---------------------------------------

dt = 0.001  # サンプリング間隔 (1ms)
success_count = 0
error_count = 0

print(f"🚀 フーリエ変換を開始します...")
print(f"📁 画像の保存先フォルダ:\n  {output_dir_path}\n")

for file_path in target_files:
    dirpath = os.path.dirname(file_path)
    csv_file = os.path.basename(file_path)
    
    # ファイル名衝突を防ぐため、末端フォルダの名前を取得します
    last_folder_name = os.path.basename(dirpath)
    
    try:
        # 1. データの読み込み (3行目がカラム名、4行目の単位は削除)
        df = pd.read_csv(file_path, header=2)
        df = df.drop(0).reset_index(drop=True)
        
        # 列のチェック
        if 'CH03' not in df.columns or 'CH04' not in df.columns:
            print(f" [スキップ] 対象列(CH03/CH04)なし: {csv_file}")
            error_count += 1
            continue
            
        df['CH03'] = pd.to_numeric(df['CH03'], errors='coerce')
        df['CH04'] = pd.to_numeric(df['CH04'], errors='coerce')
        df = df.dropna(subset=['CH03', 'CH04'])
        
        N = len(df)
        if N == 0:
            print(f" [スキップ] 有効な数値データなし: {csv_file}")
            error_count += 1
            continue

        # 2. FFTの計算
        y3 = df['CH03'].values
        y4 = df['CH04'].values

        F3 = np.fft.fft(y3)
        F4 = np.fft.fft(y4)

        # 振幅の正規化
        amp3 = np.abs(F3) / (N / 2)
        amp4 = np.abs(F4) / (N / 2)
        amp3[0] /= 2
        amp4[0] /= 2

        freq = np.fft.fftfreq(N, d=dt)
        mask = (freq >= 0)
        freq_pos, amp3_pos, amp4_pos = freq[mask], amp3[mask], amp4[mask]

        # 3. グラフの描画
        plt.figure(figsize=(10, 5))
        plt.plot(freq_pos, amp3_pos, label='CH03', alpha=0.8, linewidth=1.0)
        plt.plot(freq_pos, amp4_pos, label='CH04', alpha=0.8, linewidth=1.0)

        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Amplitude [MPa]')
        plt.title(f'FFT Spectrum - {csv_file}')
        plt.xlim(0, 500)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()

        # 4. 画像の保存設定（プログラムと同じフォルダ内の「fft_results」フォルダへ）
        # ファイル名は「[末端フォルダ名]_[CSVファイル名]_fft.png」にして衝突を防ぎます
        csv_name_without_ext = os.path.splitext(csv_file)[0]
        output_image_name = f"[{last_folder_name}]_{csv_name_without_ext}_fft.png"
        output_image_path = os.path.join(output_dir_path, output_image_name)
        
        plt.savefig(output_image_path, dpi=150)
        
        # メモリ解放
        plt.close()
        
        success_count += 1
        
    except Exception as e:
        print(f" [エラー] {csv_file}: {e}")
        error_count += 1

print("\n" + "=" * 40)
print("🏁 すべての処理が完了しました！")
print(f"✅ 成功: {success_count} 件 (画像フォルダに集約完了)")
print(f"⚠️ スキップ/エラー: {error_count} 件")
print("=" * 40)

🚀 フーリエ変換を開始します...
📁 画像の保存先フォルダ:
  c:\Users\0uh2j\OneDrive\Desktop\vscodeで\フーリエ変換\fft_results

 [エラー] 070_210℃_200_1.0mm.csv: No columns to parse from file

🏁 すべての処理が完了しました！
✅ 成功: 1289 件 (画像フォルダに集約完了)
⚠️ スキップ/エラー: 1 件
